# XGBoost

- Read in tabular data
- Define predictor and target variables
- Train XGBoost model

In [1]:
import xgboost as xgb

from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import sys
import yaml

sys.path.append('/home/548/cd3022/repos/solar-nowcast/modules')
import data_transform
from xgb_preprocessing import prepare_data

from sklearn.metrics import mean_squared_error

In [2]:
# READ PARQUET TABULAR DATA
data_path = Path('/scratch/er8/cd3022/xgb_datasets/')
df = pd.concat(
    pd.read_parquet(f)
    for f in data_path.glob('*all_training*')
)

# ADD MONTH COLUMN TO USE FOR TRAIN/TEST SPLIT
# df['month'] = df.index.get_level_values('time').month

In [3]:
df.columns

Index(['surface_global_irradiance', 'cloud_optical_depth', 'solar_elevation',
       'channel_0003_scaled_radiance', 'channel_0004_scaled_radiance',
       'channel_0005_scaled_radiance', 'channel_0007_brightness_temperature',
       'channel_0008_brightness_temperature',
       'channel_0009_brightness_temperature',
       'channel_0011_brightness_temperature',
       'channel_0013_brightness_temperature',
       'channel_0015_brightness_temperature', 'huss', 'hus850', 'hus700',
       'hus500', 'psl', 'tas', 'ta850', 'ta700', 'ta500', 'RH24mean', 'MUEL',
       'FZL', 'MULCL', 'dp850', 'dp700', 'dp500', 'KI', 'TCD', 'CCD', 'ATP',
       'cloud_optical_depth_t1', 'cloud_optical_depth_t2',
       'cloud_optical_depth_t3', 'cloud_optical_depth_t4',
       'cloud_optical_depth_t5', 'cloud_optical_depth_t6'],
      dtype='object')

In [4]:
# Based off NWCSAF cloud retrieval algorithm requirements
df['channel_0013_0015_difference'] = df['channel_0013_brightness_temperature'] - df['channel_0015_brightness_temperature']
df['channel_0011_0013_difference'] = df['channel_0011_brightness_temperature'] - df['channel_0013_brightness_temperature']
df['channel_0007_0013_difference'] = df['channel_0007_brightness_temperature'] - df['channel_0013_brightness_temperature']

In [5]:
with open("/home/548/cd3022/repos/solar-nowcast/configs/mouse/basic.yaml") as f:
    config = yaml.safe_load(f)

model_name = config["model"]["name"]
forecast_lead = config["model"]["forecast_lead"]
test_months = config["model"]["test_months"]
X_vars = config["data"]["predictors"]
target = config["data"]["target"]
y_var = f'{target}_t{forecast_lead}'

In [6]:
X_train, X_test, y_train, y_test = prepare_data(
    df=df,
    X=X_vars,
    y=y_var,
    test_months=test_months
)

In [14]:
# APPLY LOG TRANSFORM TO CLOUD OPTICAL DEPTH

# X_train['cloud_optical_depth'] = data_transform.log_transform(X_train['cloud_optical_depth'])
# X_test['cloud_optical_depth']  = data_transform.log_transform(X_test['cloud_optical_depth'])

y_train_log = data_transform.log_transform(y_train)
y_test_log  = data_transform.log_transform(y_test)

In [15]:
# DEFINE MODEL
model = xgb.XGBRegressor(
    random_state=42,
    n_estimators=2000,
    early_stopping_rounds=50,
    learning_rate=0.03,
    eval_metric='rmse',
    # device='cuda'
)

# TRAINING
model.fit(
    X_train, y_train_log,
    eval_set=[(X_train, y_train_log), (X_test, y_test_log)],
    verbose=True
)

[0]	validation_0-rmse:2.17582	validation_1-rmse:2.11923
[1]	validation_0-rmse:2.14065	validation_1-rmse:2.09040
[2]	validation_0-rmse:2.10666	validation_1-rmse:2.06298
[3]	validation_0-rmse:2.07405	validation_1-rmse:2.03694
[4]	validation_0-rmse:2.04317	validation_1-rmse:2.01154
[5]	validation_0-rmse:2.01332	validation_1-rmse:1.98772
[6]	validation_0-rmse:1.98504	validation_1-rmse:1.96489
[7]	validation_0-rmse:1.95795	validation_1-rmse:1.94372
[8]	validation_0-rmse:1.93180	validation_1-rmse:1.92333
[9]	validation_0-rmse:1.90678	validation_1-rmse:1.90348
[10]	validation_0-rmse:1.88291	validation_1-rmse:1.88538
[11]	validation_0-rmse:1.86028	validation_1-rmse:1.86759
[12]	validation_0-rmse:1.83850	validation_1-rmse:1.85113
[13]	validation_0-rmse:1.81782	validation_1-rmse:1.83528
[14]	validation_0-rmse:1.79791	validation_1-rmse:1.82011
[15]	validation_0-rmse:1.77903	validation_1-rmse:1.80631
[16]	validation_0-rmse:1.76101	validation_1-rmse:1.79298
[17]	validation_0-rmse:1.74373	validation

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,'rmse'


In [16]:
# Predict on test set
y_pred = model.predict(X_test)

# Quick evaluation of model performance using correlation and RMSE
correlation = np.corrcoef(y_test_log['cloud_optical_depth_t1'], y_pred)[0, 1]
rmse = np.sqrt(mean_squared_error(y_test_log['cloud_optical_depth_t1'], y_pred))

print(f"Correlation between predicted and actual values: {correlation:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")

Correlation between predicted and actual values: 0.681
Root Mean Squared Error (RMSE): 1.579


In [ ]:
model.save_model(f"/scratch/er8/cd3022/xgb_models/{model_name}.json")

# Validation

In [ ]:
results = model.evals_result()

train_loss = results['validation_0']['rmse']
val_loss = results['validation_1']['rmse']

plt.figure()
plt.plot(train_loss, label='Train RMSE')
plt.plot(val_loss, label='Validation RMSE')

plt.xlabel('Boosting Iterations')
plt.ylabel('RMSE')
plt.title('Training vs Validation Loss')
plt.legend()

plt.show()

In [ ]:
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(6, 6))
ax[0,0].hist(y_train_log, bins=100)
ax[0,0].set_title("True-train")
ax[0,0].set_xlim(-3, 5)
ax[0,0].set_ylim(0, 800_000)

ax[0,1].hist(y_pred_train, bins=100)
ax[0,1].set_title("Prediction-train")
ax[0,1].set_xlim(-3, 5)
ax[0,1].set_ylim(0, 800_000)

ax[1,0].hist(y_test_log, bins=100)
ax[1,0].set_title("True-test")
ax[1,0].set_xlim(-3, 5)
ax[1,0].set_ylim(0, 800_000)

ax[1,1].hist(y_pred_test, bins=100)
ax[1,1].set_title("Prediction-test")
ax[1,1].set_xlim(-3, 5)
ax[1,1].set_ylim(0, 800_000)